In [0]:
%sql
use catalog training_dev;

#### Setup
- create volume
- create folder within volume
- upload file to volume

In [0]:
%sql
create volume if not exists training_dev.default.raw;

In [0]:
%fs
mkdirs /Volumes/training_dev/default/raw/order_hist

In [0]:
%fs
ls /Volumes/training_dev/default/raw

In [0]:
dbutils.fs.ls('/Volumes/training_dev/default/raw')

In [0]:
%sql
list '/Volumes/training_dev/default/raw/order_hist'

In [0]:
%sql
select * from csv.`dbfs:/Volumes/training_dev/default/raw/order_hist`

In [0]:
%sql
select * from read_files('dbfs:/Volumes/training_dev/default/raw/order_hist',format=>'csv')
limit 3;

##### Ingestion
- Initial Load: Use CTAS

In [0]:
%sql
create or replace table order_m as
select * from read_files('dbfs:/Volumes/training_dev/default/raw/order_hist',format=>'csv')
;

In [0]:
%sql
select * from order_m;

##### Upload another file and run CTAS again
- no incremental processing

In [0]:
%sql
create or replace table order_m as
select * from read_files('dbfs:/Volumes/training_dev/default/raw/order_hist',format=>'csv')
;

select * from order_m;

##### Upload another file, with an extra column and run CTAS again
- no incremental processing
- extra column is reflected in the table

In [0]:
%sql
create or replace table order_m as
select * from read_files('dbfs:/Volumes/training_dev/default/raw/order_hist',format=>'csv')
;

select * from order_m;

##### Upload another file, with an incorrect datatype and run CTAS again
- no incremental processing
- string values in integer field is infers , and changes data type to string

In [0]:
%sql
create or replace table order_m as
select * from read_files('dbfs:/Volumes/training_dev/default/raw/order_hist',format=>'csv')
;

select * from order_m;

##### Schema enforcement

In [0]:
%sql
create or replace table order_m as
select * 
from read_files(
        'dbfs:/Volumes/training_dev/default/raw/order_hist',
         format=>'csv',
         schema => 'order_id int, order_date timestamp, customer_id string, order_status string, order_amount bigint, _rescued_data string',
         header => true,
         mode => 'PERMISSIVE',
         columnNameOfCorruptRecord => '_rescued_data'     
              )
;

select * from order_m;